In [1]:
import pandas as pd

In [2]:
datasets_count = 1500

In [3]:
source_df = pd.read_csv('../datasets/links_sipora.csv')
endpoints: dict[int, None | str] = {}

for source_df in source_df.itertuples():
    if endpoints.get(source_df.year) is None:
        endpoints[source_df.year] = source_df.link
    else:
        continue

print(endpoints)

{2016: 'https://sipora.polije.ac.id/53207/', 2017: 'https://sipora.polije.ac.id/16352/', 2018: 'https://sipora.polije.ac.id/23207/', 2019: 'https://sipora.polije.ac.id/23546/', 2020: 'https://sipora.polije.ac.id/2648/'}


In [4]:
df = pd.read_csv("../datasets/contents_sipora_human.csv")
df.head()

,link,content,type,lang,label
0,https://sipora.polije.ac.id/53207/,Bayam merupakan tumbuhan yang biasa ditanam un...,abstract,id,human
1,https://sipora.polije.ac.id/53207/,Kue kering merupakan salah satu jenis dari mak...,abstract,id,human
2,https://sipora.polije.ac.id/53207/,Pemasaran produk kue kering bayam dilakukan s...,abstract,id,human
3,https://sipora.polije.ac.id/53207/,Bayam merupakan tumbuhan yang biasa ditanam un...,content,id,human
4,https://sipora.polije.ac.id/53207/,Bayam juga sangat kaya akan antioksidan baik u...,content,id,human


In [5]:
filtered_data = None
count_per_year = int(datasets_count / len(endpoints))
column = ['link', 'content', 'type', 'lang', 'label']

for year in endpoints:
    endpoint = endpoints[year]
    index = df[df['link'] == endpoint].index
    filtered_items = df.iloc[index[0]:index[0]+count_per_year]
    
    if filtered_data is None:
        filtered_data = filtered_items
    else:
        filtered_data = pd.concat([filtered_data, filtered_items], ignore_index=True)

df = filtered_data

In [6]:
from pathlib import Path
import json
import re

def create_datasets(df: pd.DataFrame, dtype = 'human'):
    output_dir = f"../datasets/cleaned/{dtype}/"
    dir = Path(output_dir)
    dir.mkdir(parents=True, exist_ok=True)
    
    id_count: dict[str, int] = {}
    file_created = 0
    
    for item in df.itertuples():
        sentences = [];
        last_index = 0;
    
        try:
            for sentence in item.content.split("."):
                start = last_index
                end = last_index + len(sentence)
        
                if start < end:
                    sentences.append({
                        "start": start,
                        "end": end,
                        "label": dtype,
                    })
                    last_index = end + 1
                    
            doc_id = re.findall(r'[0-9]+', item.link)[0] or "unknown"
            current_doc_id_count = id_count.get(doc_id, 0)
            text_id = f"{doc_id}_{current_doc_id_count}"
            
            data_json = json.dumps({
                'text_id': text_id,
                'document_id': doc_id,
                'content': item.content,
                'type': item.type,
                'lang': item.lang,
                'label': dtype,
                'sentences': sentences,
            })
        
            filename = f"{text_id}.json"
            with open(f"{output_dir}{filename}", "w") as f:
                f.write(data_json)
                id_count[doc_id] = current_doc_id_count + 1
                file_created += 1
        except Exception as _:
            pass

    print(f"{file_created} file created!")

In [7]:
create_datasets(df, "human")

1500 file created!


In [8]:
chatgpt_df = pd.read_csv("../datasets/contents_sipora_chatgpt.csv")
claude_df = pd.read_csv("../datasets/contents_sipora_claude.csv")
gemini_df = pd.read_csv("../datasets/contents_sipora_gemini.csv")
grok_df = pd.read_csv("../datasets/contents_sipora_grok.csv")
qwen_df = pd.read_csv("../datasets/contents_sipora_qwen.csv")

ai_df = pd.concat([chatgpt_df, claude_df, gemini_df, grok_df, qwen_df], ignore_index=True)
print(ai_df.count())
ai_df.head()

create_datasets(ai_df, "ai")

link       1500
content    1500
type       1500
lang       1500
label      1500
dtype: int64
1500 file created!
